# Telecom Customer Churn: Model Selection

Load and prepare the source data, compare classifiers, select a constrained decision threshold, and export the chosen model. This notebook runs independently of the EDA notebook.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

project_root = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv").is_file()
)
file_path = project_root / "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(file_path)

sns.set_theme(style="whitegrid")
churn_palette = {"No": "#4C78A8", "Yes": "#E45756"}

df = df.drop(columns="customerID")
df["SeniorCitizen"] = df["SeniorCitizen"].map({1: "Yes", 0: "No"})
text_columns = df.select_dtypes(include="str").columns
df[text_columns] = df[text_columns].apply(lambda column: column.str.strip())
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
new_customer_missing_charge = (
    df["TotalCharges"].isna() & df["tenure"].eq(0) & df["Churn"].eq("No")
)
df.loc[new_customer_missing_charge, "TotalCharges"] = 0


In [ ]:
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
    fbeta_score,
    precision_score,
    recall_score,
    roc_curve
)
from sklearn.model_selection import (
    cross_val_predict,
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

# This is the largest allowed fraction of retained customers that may be
# incorrectly sent to the churn-retention campaign.
MAX_FALSE_POSITIVE_RATE = 0.31

def positive_class_scores(estimator, X_values):
    # Most models return churn probabilities. SVC returns decision scores.
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(X_values)[:, 1]
    return estimator.decision_function(X_values)

def recall_at_fpr_cap(estimator, X_values, y_values):
    # Compare models by the highest recall they can reach without
    # exceeding the allowed false-positive rate.
    scores = positive_class_scores(estimator, X_values)
    false_positive_rates, recalls, _ = roc_curve(y_values, scores)
    allowed_recalls = recalls[
        false_positive_rates <= MAX_FALSE_POSITIVE_RATE
    ]
    return allowed_recalls.max() if len(allowed_recalls) else 0.0

## Baseline churn model

A logistic-regression pipeline provides an interpretable baseline for predicting churn. All preprocessing is fitted inside the pipeline so that transformations learned from the training data are applied consistently during cross-validation and final testing.

In [ ]:
# Create the feature table
X = df.drop(columns=["Churn"])

# Create the target
y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

### Feature and target definition

The target is encoded as `1` for churn and `0` for retention. Predictor columns are separated by data type so categorical and numerical features can receive the appropriate preprocessing.

In [ ]:
# Get categorical features
categorical_features = (
    X
    .select_dtypes(include="str")
    .columns
    .tolist()
)


numeric_features = [
    column
    for column in X.columns
    if column not in categorical_features 
]

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numeric_features)

### Train–test split

The data is split into an 80% training set and a 20% holdout test set. Stratification preserves the original churn rate in both partitions, while `random_state=42` makes the split reproducible. The test set remains untouched until the final evaluation.

In [ ]:
# Keep 20% completely untouched for the final testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining churn distribution:")
print(
    y_train
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nTest churn distribution:")
print(
    y_test
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

### Preprocessing

Categorical variables are one-hot encoded, with previously unseen categories ignored safely at prediction time. Numerical variables are rescaled to the 0–1 range so the logistic-regression regularization treats their coefficient magnitudes comparably.

In [ ]:
preprocessor = ColumnTransformer([
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features
    ),
    (
        "numeric",
        MinMaxScaler(),
        numeric_features
    )
])

### Modeling pipeline

The preprocessing steps and logistic-regression classifier are combined into one pipeline. This design reduces data-leakage risk because every cross-validation fold learns its encodings and scaling parameters from that fold's training portion only.

In [ ]:
logistic_pipeline = Pipeline([
    (
        "preprocess",
        preprocessor
    ),
    (
        "model",
        LogisticRegression(
            solver="liblinear",
            max_iter=1000,
            random_state=42
        )
    )
])

### Cross-validation strategy

Five-fold stratified cross-validation estimates performance across multiple train–validation partitions while preserving the churn proportion in each fold. Shuffling with a fixed random seed improves reproducibility.

In [ ]:
# Splits training set for validation with shuffled combinations
cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

### Hyperparameter search space

The search expands regularization below the previous boundary winner and tests class weights around the observed imbalance. Each candidate is judged by the highest churn recall it can achieve while keeping the validation false-positive rate at or below 31%.

In [ ]:
logistic_parameter_grid = {
    # Smaller C means stronger regularization.
    "model__C": [
        0.001,
        0.003,
        0.01,
        0.03,
        0.1
    ],
    "model__l1_ratio": [
        0.0, 
        1.0
    ],
    # Class weights alter the fitted boundary; the later threshold step
    # separately controls which customers receive retention outreach.
    "model__class_weight": [
        None,
        "balanced",
        {0: 1, 1: 2.5},
        {0: 1, 1: 3.0},
        {0: 1, 1: 3.5},
        {0: 1, 1: 4.0}
    ]
}

### Model selection

Grid search evaluates every parameter combination using constrained recall. Within each validation fold, it measures the best recall available without allowing more than 31% of retained customers to become false positives. F2 is still reported later, but it no longer permits unlimited unnecessary outreach during model selection.

In [ ]:
# Choose the parameters with the best recall under the FPR cap.
logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_parameter_grid,
    scoring=recall_at_fpr_cap,
    cv=cross_validation,
    n_jobs=-1,
    verbose=1
)

logistic_search.fit(
    X_train, 
    y_train
)

The model search above uses cross-validation on the training split. The holdout test is evaluated only after the final model and threshold are fixed.


---

## Decision-tree classifier

A decision tree is evaluated as a nonlinear alternative to logistic regression. It can learn threshold-based rules and interactions between customer attributes without requiring those relationships to be specified in advance. The same training split, cross-validation folds, preprocessing, and constrained-recall objective are retained for a fair comparison.

### Modeling pipeline

The decision tree is combined with the existing preprocessor in a single pipeline, preventing information from the validation folds from influencing preprocessing. One-hot encoding is still required for categorical variables; numerical scaling is not necessary for a tree, but retaining the shared preprocessor keeps the model comparison consistent. A fixed random seed makes the tree reproducible.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_pipeline = Pipeline([
    (
        "preprocess",
        preprocessor
    ),
    (
        "model",
        DecisionTreeClassifier(
            random_state=42
        )
    )
])

### Hyperparameter search space

The expanded search tunes the tree's split rules, pruning, and sensitivity to churners:

- `max_depth` limits the number of decision levels and reduces the risk of an overly complex tree.
- `min_samples_leaf` and `min_samples_split` require customer segments to contain enough observations to form stable rules.
- `ccp_alpha` removes weak branches through cost-complexity pruning.
- `criterion` compares Gini impurity with logarithmic loss.
- `class_weight` increases the cost of misclassifying churners, encouraging higher churn recall.

Randomized search keeps the larger search affordable. Every candidate must maximize recall under the same 31% validation false-positive-rate cap.

In [ ]:
tree_parameter_distributions = {
    "model__criterion": [
        "gini",
        "log_loss"
    ],
    "model__max_depth": [
        4,
        5,
        6,
        7,
        8,
        10,
        None
    ],
    "model__min_samples_leaf": [
        10,
        15,
        20,
        30,
        40,
        60
    ],
    "model__min_samples_split": [
        30,
        50,
        75,
        100,
        150
    ],
    "model__ccp_alpha": [
        0.0,
        0.00001,
        0.00003,
        0.0001,
        0.0003,
        0.001,
        0.003,
        0.01
    ],
    "model__class_weight": [
        None,
        "balanced",
        {0: 1, 1: 2.5},
        {0: 1, 1: 3.0},
        {0: 1, 1: 3.5},
        {0: 1, 1: 4.0},
        {0: 1, 1: 4.5},
        {0: 1, 1: 5.0}
    ]
}

### Model selection

Randomized search evaluates 80 combinations using the same five-fold stratified cross-validation strategy. It selects the tree with the highest validation recall available under the 31% false-positive-rate cap. Training scores are retained to help diagnose overfitting.

In [ ]:
tree_search = RandomizedSearchCV(
    estimator=tree_pipeline,
    param_distributions=tree_parameter_distributions,
    n_iter=80,
    scoring=recall_at_fpr_cap,
    cv=cross_validation,
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
    verbose=1
)

tree_search.fit(
    X_train,
    y_train
)

The model search above uses cross-validation on the training split. The holdout test is evaluated only after the final model and threshold are fixed.


---

## Random-forest classifier

A random forest is evaluated to determine whether an ensemble of decision trees can improve churn detection. Each tree is trained on a sample and considers a random subset of features when splitting, which generally reduces the variance and overfitting risk of a single decision tree. The same data split, preprocessing, cross-validation folds, and constrained-recall objective are retained for comparability.

### Modeling pipeline

The random forest is placed inside the existing preprocessing pipeline so that categorical encoding is learned independently within each cross-validation fold. As with the decision tree, numerical scaling is not required by the model, but retaining the shared preprocessor ensures a consistent comparison.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest_pipeline = Pipeline([
    (
        "preprocess",
        preprocessor
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=400,
            bootstrap=True,
            random_state=42
        )
    )
])

### Hyperparameter search space

The expanded search tunes tree complexity, tree diversity, bootstrap sampling, and sensitivity to churners:

- `max_depth` limits the complexity of each tree and helps prevent overfitting.
- `min_samples_leaf` and `min_samples_split` smooth highly specific rules.
- `max_features` controls how many transformed features are available at each split.
- `max_samples` changes the bootstrap sample size and therefore the diversity between trees.
- `class_weight` increases the cost of missing churners. `balanced_subsample` recalculates automatic class weights separately for each bootstrap sample.

The forest uses 400 trees for stable comparisons while randomized search evaluates 80 structural combinations. The selected candidate must maximize recall without exceeding the 31% validation false-positive-rate cap.

In [ ]:
forest_parameter_distributions = {
    "model__criterion": [
        "gini",
        "log_loss"
    ],
    "model__max_depth": [
        4,
        5,
        6,
        8,
        None
    ],
    "model__min_samples_leaf": [
        5,
        10,
        15,
        20,
        30
    ],
    "model__min_samples_split": [
        2,
        10,
        20,
        40
    ],
    "model__max_features": [
        "sqrt",
        "log2",
        0.3,
        0.5,
        0.7
    ],
    "model__max_samples": [
        0.65,
        0.8,
        None
    ],
    "model__class_weight": [
        None,
        "balanced",
        "balanced_subsample",
        {0: 1, 1: 2.5},
        {0: 1, 1: 3.0},
        {0: 1, 1: 3.5},
        {0: 1, 1: 4.0},
        {0: 1, 1: 4.5}
    ]
}

### Model selection

Randomized search evaluates 80 combinations with five-fold stratified cross-validation. It compares forests by recall under the common 31% false-positive-rate cap instead of rewarding a model for sending substantially more customers to retention.

In [ ]:
forest_search = RandomizedSearchCV(
    estimator=forest_pipeline,
    param_distributions=forest_parameter_distributions,
    n_iter=80,
    scoring=recall_at_fpr_cap,
    cv=cross_validation,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

forest_search.fit(
    X_train,
    y_train
)

The model search above uses cross-validation on the training split. The holdout test is evaluated only after the final model and threshold are fixed.


## LightGBM classifier

LightGBM is evaluated as a gradient-boosting model. Unlike the random forest, which builds independent trees, LightGBM builds trees sequentially so that each new tree focuses on errors made by the previous trees. This can capture nonlinear relationships efficiently. The same data split, preprocessing, cross-validation folds, and constrained-recall objective are retained for a fair comparison.

### Modeling pipeline

LightGBM is placed inside the existing preprocessing pipeline. Categorical features are one-hot encoded within each cross-validation fold, while the numerical features use the same scaling as the earlier models. LightGBM does not require scaling, but reusing the preprocessor keeps the input data consistent across all models.

In [ ]:
from lightgbm import LGBMClassifier

lightgbm_pipeline = Pipeline([
    (
        "preprocess",
        preprocessor
    ),
    (
        "model",
        LGBMClassifier(
            objective="binary",
            random_state=42,
            n_jobs=1,
            verbosity=-1
        )
    )
])

### Hyperparameter search space

The earlier LightGBM search selected values at several grid boundaries: the most trees, the smallest learning rate, the fewest leaves, and the largest churn weight. The expanded search therefore explores farther in those directions and adds regularization controls:

- `n_estimators` and `learning_rate` jointly control how gradually the model learns.
- `num_leaves`, `max_depth`, and `min_child_samples` control tree complexity.
- `reg_alpha` and `reg_lambda` add L1 and L2 regularization.
- `colsample_bytree`, `subsample`, and `subsample_freq` introduce feature and customer sampling.
- `scale_pos_weight` increases the cost of missing churners in this binary-classification problem.

The complete combination set would be unnecessarily large, so randomized search tests 60 reproducible combinations.

In [ ]:
lightgbm_parameter_distributions = {
    "model__n_estimators": [
        200,
        300,
        500,
        800
    ],
    "model__learning_rate": [
        0.01,
        0.02,
        0.03,
        0.05
    ],
    "model__num_leaves": [
        3,
        5,
        7,
        10,
        15
    ],
    "model__max_depth": [
        3,
        4,
        5,
        -1
    ],
    "model__min_child_samples": [
        10,
        20,
        40,
        60
    ],
    "model__reg_alpha": [
        0,
        0.1,
        0.5,
        1
    ],
    "model__reg_lambda": [
        0,
        0.1,
        0.5,
        1
    ],
    "model__colsample_bytree": [
        0.7,
        0.85,
        1.0
    ],
    "model__subsample": [
        0.7,
        0.85,
        1.0
    ],
    "model__subsample_freq": [
        1
    ],
    "model__scale_pos_weight": [
        1,
        2,
        2.5,
        3,
        3.5,
        4
    ]
}

### Model selection

Randomized search evaluates 60 LightGBM parameter combinations using the same five-fold stratified cross-validation. Candidates are compared by recall under the 31% validation false-positive-rate cap, and `random_state=42` makes the sampled combinations reproducible.

In [ ]:
lightgbm_search = RandomizedSearchCV(
    estimator=lightgbm_pipeline,
    param_distributions=lightgbm_parameter_distributions,
    n_iter=60,
    scoring=recall_at_fpr_cap,
    cv=cross_validation,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

lightgbm_search.fit(
    X_train,
    y_train
)

The model search above uses cross-validation on the training split. The holdout test is evaluated only after the final model and threshold are fixed.


## Support-vector-machine classifier

A support vector machine (SVM) is evaluated with both linear and radial-basis-function (RBF) boundaries. The same split, fold-safe preprocessing, cross-validation folds, and constrained-recall objective are retained.

### Modeling pipeline

SVMs are sensitive to feature scale. The search therefore compares the existing minimum–maximum scaling with standardization while fitting both transformations inside every cross-validation fold.

In [ ]:
from sklearn.svm import SVC

svm_standard_preprocessor = ColumnTransformer([
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features
    ),
    (
        "numeric",
        StandardScaler(),
        numeric_features
    )
])

svm_pipeline = Pipeline([
    (
        "preprocess",
        preprocessor
    ),
    (
        "model",
        SVC(cache_size=500)
    )
])

### Hyperparameter search space

The expanded search tunes the SVM boundary and preprocessing:

- `C` controls the penalty for classification errors. A larger value tries harder to classify the training examples correctly but may overfit.
- `gamma` controls how far each training example influences the RBF boundary; it is omitted for the linear kernel.
- `class_weight` increases the cost of missing churners.
- `preprocess` compares minimum–maximum scaling with standardization.

Separate parameter dictionaries prevent irrelevant `gamma` values from being tested with the linear kernel. Randomized search evaluates 80 combinations under the common 31% validation false-positive-rate cap.

In [ ]:
svm_class_weights = [
    None,
    "balanced",
    {0: 1, 1: 1.5},
    {0: 1, 1: 2.0},
    {0: 1, 1: 2.5},
    {0: 1, 1: 3.0},
    {0: 1, 1: 3.5},
    {0: 1, 1: 4.0}
]

svm_parameter_distributions = [
    {
        "preprocess": [preprocessor, svm_standard_preprocessor],
        "model__kernel": ["linear"],
        "model__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1],
        "model__class_weight": svm_class_weights
    },
    {
        "preprocess": [preprocessor, svm_standard_preprocessor],
        "model__kernel": ["rbf"],
        "model__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1],
        "model__gamma": [
            0.0001,
            0.0003,
            0.001,
            0.003,
            0.01,
            0.03,
            "scale"
        ],
        "model__class_weight": svm_class_weights
    }
]

### Model selection

Randomized search evaluates 80 reproducible SVM combinations with five-fold stratified cross-validation and selects the combination with the highest recall under the 31% false-positive-rate cap.

In [ ]:
svm_search = RandomizedSearchCV(
    estimator=svm_pipeline,
    param_distributions=svm_parameter_distributions,
    n_iter=80,
    scoring=recall_at_fpr_cap,
    cv=cross_validation,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

svm_search.fit(
    X_train,
    y_train
)

The model search above uses cross-validation on the training split. The holdout test is evaluated only after the final model and threshold are fixed.


## Select the model using training cross-validation only

The search score is recall under the 31% false-positive-rate cap. Select the highest validation score; if scores differ by at most 0.001, prefer Random Forest for its established application contract. SVM is reported as a comparison but is not exportable because the application expects probability scores. The test split remains untouched at this point.


In [ ]:
searches = {
    "Random Forest": forest_search,
    "LightGBM": lightgbm_search,
    "Logistic Regression": logistic_search,
    "Decision Tree": tree_search,
}
validation_scores = pd.Series({name: search.best_score_ for name, search in searches.items()})
best_validation_score = validation_scores.max()
preference = ["Random Forest", "LightGBM", "Logistic Regression", "Decision Tree"]
winner_name = next(name for name in preference if validation_scores[name] >= best_validation_score - 0.001)
winning_search = searches[winner_name]
final_model = winning_search.best_estimator_
print(validation_scores.sort_values(ascending=False))
print("SVM training CV score (comparison only):", round(svm_search.best_score_, 3))
print("Selected model:", winner_name)


## Choose the decision threshold using training predictions

Out-of-fold scores for the selected estimator define the highest-recall threshold that stays under the training FPR cap. This threshold is frozen before the holdout test is opened.


In [ ]:
oof_scores = cross_val_predict(
    final_model, X_train, y_train, cv=cross_validation, method="predict_proba", n_jobs=-1
)[:, 1]
training_fprs, training_recalls, candidate_thresholds = roc_curve(y_train, oof_scores)
allowed_positions = np.flatnonzero(training_fprs <= MAX_FALSE_POSITIVE_RATE)
best_position = allowed_positions[np.argmax(training_recalls[allowed_positions])]
final_threshold = float(candidate_thresholds[best_position])
print("Training OOF recall:", round(training_recalls[best_position], 3))
print("Training OOF FPR:", round(training_fprs[best_position], 3))
print("Frozen threshold:", round(final_threshold, 4))


## Final holdout evaluation

Evaluate the selected model and frozen threshold once. Holdout results describe generalization; they do not change the model or threshold.


In [ ]:
test_scores = final_model.predict_proba(X_test)[:, 1]
test_predictions = (test_scores >= final_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, test_predictions, labels=[0, 1]).ravel()
final_test_metrics = {
    "recall": recall_score(y_test, test_predictions),
    "precision": precision_score(y_test, test_predictions, zero_division=0),
    "f2": fbeta_score(y_test, test_predictions, beta=2),
    "fpr": fp / (fp + tn),
    "false_positives": int(fp),
    "false_negatives": int(fn),
    "customers_flagged": int(tp + fp),
}
print(pd.Series(final_test_metrics))
print(classification_report(y_test, test_predictions, target_names=["Stayed", "Churned"], digits=3))
ConfusionMatrixDisplay.from_predictions(y_test, test_predictions, labels=[0, 1], display_labels=["Stayed", "Churned"], cmap="Blues", values_format="d")
plt.title(f"{winner_name}: frozen-threshold holdout confusion matrix")
plt.show()


## Export a candidate bundle

Save to `models/candidates/` so rerunning the notebook cannot replace the trusted model used by the API. To promote a candidate, review its test results, update the application's model-version and threshold contract, configure its path and SHA-256, and rerun application tests. A changed checksum alone is not sufficient for this application's strict loader.


In [ ]:
import hashlib
import joblib
import sklearn

model_directory = project_root / "models" / "candidates"
model_directory.mkdir(parents=True, exist_ok=True)
model_path = model_directory / f"{winner_name.lower().replace(' ', '_')}_churn_bundle.joblib"
bundle = {
    "bundle_version": 1,
    "pipeline": final_model,
    "threshold": final_threshold,
    "positive_class": 1,
    "target_mapping": {"No": 0, "Yes": 1},
    "feature_columns": X_train.columns.tolist(),
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "selected_parameters": winning_search.best_params_,
    "training_fpr_cap": MAX_FALSE_POSITIVE_RATE,
    "test_metrics": final_test_metrics,
    "sklearn_version": sklearn.__version__,
}
joblib.dump(bundle, model_path, compress=3)
loaded = joblib.load(model_path)
assert np.allclose(loaded["pipeline"].predict_proba(X_test.head()), final_model.predict_proba(X_test.head()))
print("Candidate:", model_path)
print("SHA-256:", hashlib.sha256(model_path.read_bytes()).hexdigest())
